
Notebook for the initial testing of features

In [3]:
import httpx
import csv

### HTTP Data Extraction

In [4]:
url = f"https://repodatos.atdt.gob.mx/api_update/conanp/areas_destinadas_voluntariamente_conservacion/relacion_advc.csv"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/156.0.0.0 Safari/537.36",
    "Referer": "https://www.datos.gob.mx/",
    "Accept": "text/csv,*/*;q=0.8",
    "Accept-Language": "es-MX,es;q=0.9,en;q=0.8",
}

In [9]:
with httpx.Client(headers=headers, follow_redirects=True, timeout=60, http2= True) as client:
    with client.stream("GET", url) as r:
        r.read()   # ← required
        print(r.status_code)
        data = r.json()['result']['records']
        print(data)
        fieldnames = list(set().union(*data))
        with open(f"d.csv", "wt", newline='', encoding='utf-8') as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames, restval='') 
            writer.writeheader()
            writer.writerows(data)


200


JSONDecodeError: Expecting value: line 1 column 1 (char 0)

### Data processing

In [10]:
import os

In [27]:
"_".join('notas_impacto_ambiental_resolved_20261119162428'.split('_')[0:-1])

'notas_impacto_ambiental_resolved'

In [23]:
files = os.listdir('../grdata/_raw')

for file in files:
    print(file)
    if file in ['advc_20261118145643.csv','areas_protegidas_20261118151600.csv']:
        os.rename('../grdata/_raw' + '/' + file, '../grdata/_processed' + '/' + file)
    else:
        continue

advc_20261118145643.csv
advc_20261118151600.csv
advc_20261118152946.csv
advc_20261118153122.csv
advc_20261119161353.csv
advc_20261119161426.csv
advc_20261119161503.csv
areas_protegidas_20261118151600.csv
areas_protegidas_20261118152946.csv
areas_protegidas_20261119161503.csv
biosferas_UNESCO_20261118151600.csv
biosferas_UNESCO_20261118152946.csv
biosferas_UNESCO_20261119161503.csv
generacion_res_pel_tipo_20261118151600.csv
generacion_res_pel_tipo_20261118152946.csv
generacion_res_pel_tipo_20261119161503.csv
historico_incendios_forestales_20261118151600.csv
historico_incendios_forestales_20261118152946.csv
historico_incendios_forestales_20261119161503.csv
incendios_forestales_20261118151600.csv
incendios_forestales_20261118152946.csv
incendios_forestales_20261119161503.csv
notas_impacto_ambiental_20261118152946.csv
notas_impacto_ambiental_20261119161503.csv
notas_impacto_ambiental_resolved_20261118152946.csv
notas_impacto_ambiental_resolved_20261119161503.csv
patrimonios_20261118151600.

### Table readings

In [31]:
import polars as pl

In [41]:
tables = os.listdir('../grdata/_processed')

for i, table in enumerate(tables, 1):
    print(i, table)
    #print(pl.read_parquet('../grdata/_processed/' + table))
    print("""************************************************************""")


1 advc.parquet
************************************************************
2 areas_protegidas.parquet
************************************************************
3 biosferas_UNESCO.parquet
************************************************************
4 generacion_res_pel_tipo.parquet
************************************************************
5 historico_incendios_forestales.parquet
************************************************************
6 incendios_forestales.parquet
************************************************************
7 notas_impacto_ambiental.parquet
************************************************************
8 notas_impacto_ambiental_resolved.parquet
************************************************************
9 patrimonios.parquet
************************************************************
10 sitios_cont.parquet
************************************************************
11 sitios_cont_tipo.parquet
************************************************************
12 s

### Protected Zones Transformation

In [36]:
import polars as pl
from IPython.display import Markdown, display, HTML


l = ['advc.parquet', 'areas_protegidas.parquet', 'biosferas_UNESCO.parquet', 'patrimonios.parquet']
PATH = '../grdata/_processed/'

for file in l:
    table = pl.read_parquet(PATH + file).head(1)
    print(file)
    from IPython.display import Markdown, display

    html = '<div style="display:flex; gap:24px;">'
    html += f'<div>{table._repr_html_()}</div>'
    html += '</div>'
    display(HTML(html))


advc.parquet


numero,numero_certificado,nombre_area,superficie_certificada_ha,municipio,estado,anio_certificacion,vigencia_anio,tipo_propietario,region_conanp
i64,str,str,f64,str,str,i64,f64,str,str
1,"""CONANP-01/2002""","""Parque Ecológico Jaguaroundi""",960.734884,"""Coatzacoalcos""","""Veracruz""",2002,99.0,"""Empresa de Participación Estat…","""Planicie Costera y Golfo de Mé…"


areas_protegidas.parquet


sup_ha,fecha_dec,fecha_pm,num_anp,ficha_simec,region,cat_manejo,sterr_ha,nom_direc,cert_sinap,cui_dof,fecha_otro,smar_ha,estado,cui_anp,_id,nombre,desig_inter
f64,str,str,f64,str,str,str,f64,str,str,str,str,f64,str,str,i64,str,str
934756.25,"""1993-06-10T00:00:00""","""2009-09-25T00:00:00""",110.0,"""https://simec.conanp.gob.mx/fi…","""Noroeste y Alto Golfo de Calif…","""RB""",407147.5456,"""M. en A. Domingo de Jesús Zata…","""SINAP 004""","""2.1.01.110""","""1993-06-10T00:00:00""",527608.7044,"""Baja California y Sonora""","""ANP00201110""",1,"""Alto Golfo de California y Del…","""World Heritage - Patrimonio Mu…"


biosferas_UNESCO.parquet


estado_que_abarca,numero,region,reserva_biosfera_mab,nombre_director,anio_designacion,superficie_total_ha,anp_involucrada_rbmab,_id,anio_modificacion_superficie
str,i64,str,str,str,i64,f64,str,i64,str
"""Coahuila-Durango-Chihuahua""",1,"""Noreste y Sierra Madre Orienta…","""Mapimi""","""Juan Carlos Ibarra Flores""",1977,342387.0,"""Mapimi""",1,"""sin dato"""


patrimonios.parquet


numero,categoria_manejo_anp,lista_patrimonio_mundial_peligro,nombre_director,tipo_sitio,fecha_inscripcion,estado,criterios_inscripcion,_id,sitio_patrimonio_mundial,anp_involucrada_sitio,superficie_total_(ha),url_patrimonio_mundial
i64,str,str,str,str,i64,str,str,i64,str,str,f64,str
1,"""Reserva de Biosfera""","""No""","""Oscar Guzmán Escalante""","""Natural""",1987,"""Quintana Roo""","""VII-X""",1,"""Sian Kaan""","""Sian Kaan""",528000.0,"""https://whc.unesco.org/en/list…"


In [39]:
pl.read_parquet(PATH + 'advc.parquet').columns

['numero',
 'numero_certificado',
 'nombre_area',
 'superficie_certificada_ha',
 'municipio',
 'estado',
 'anio_certificacion',
 'vigencia_anio',
 'tipo_propietario',
 'region_conanp']

##### Dictionary of variables:
| Column | advc | areas_protegidas | biosferas_UNESCO | patrimonios |
| -------- | -------- | -------- | -------- | -------- |
| municipality | municipio | Null | Null | Null |
| state | estado | estado | estado_que_abarca |estado|
| place_name | nombre_area | nombre | reserva_biosfera_mab | sitio_patrimonio_mundial |
| owner_type | tipo_propietario | desig_inter | Null | Null | 
| certification | numero_certificado | cert_sinap o cui_anp | Null | Null |
| owner | tipo_propietario | nom_direc | nombre_director | nombre_director |
| is_endangered? | Null | Null | Null | Null |  


In [48]:
import json

def change_col_names(df: pl.DataFrame, old_name: str, new_name: str) -> pl.DataFrame:

    return df.rename({old_name: new_name}, strict=False)

with open("../src/data_mapping.json") as mappings_file:
    mappings = json.load(mappings_file)

frames = {name: pl.read_parquet(PATH + name + '.parquet') for name in mappings}
# print(frames)

for name in mappings:
    df = frames[name][0]
    for old, new in mappings[name].items():
        # print(name)
        # print(type(old))
        # print(type(frames[name][0]))
        df = change_col_names(df = df, old_name = old, new_name = new)
    # print(df.columns)
    f = df.select( ["municipality",
        "state",
        "place_name",
        "owner_type",
        "certification",
        "owner",
        "is_endangered?"]
    )
    print(f)
     
    


ColumnNotFoundError: unable to find column "owner"; valid columns: ["numero", "certification", "place_name", "superficie_certificada_ha", "municipality", "state", "anio_certificacion", "vigencia_anio", "owner_type", "region_conanp"]